# Model development, iteration and trial log

This notebook documents the work that led to the final analysis, including unsuccessful trials and cases where additional optimisation did not improve the original model. The final result was not produced by fitting one model once. It emerged from repeated screening, extension, tuning, checking and comparison.

The project distinguishes the **frozen comparison** from later **exploratory development**. This allows the full modelling effort to be visible without presenting post-hoc improvements as confirmatory evidence.


## 1. Development story at a glance

The modelling process followed six stages:

1. **Establish two physical baselines.** Velocity persistence and spatial mean flow set realistic no-learning reference levels.
2. **Run a deliberately broad first screening.** We initially compared four learned models: pooled VAR(1), Ridge, weighted KNN and FNN. They were chosen to represent dynamic-linear, regularised-linear, analogue-based and neural-network approaches.
3. **Keep the promising ideas and widen the search.** The first round showed that flexible multi-output models were much stronger than pooled VAR(1), so we refined FNN and weighted KNN and added Extra Trees and Random Forest.
4. **Optimise promising models separately.** Architecture, regularisation, neighbour weighting, feature treatment and ensemble settings were tested using grouped training-only cross-validation.
5. **Retain negative results.** More tuning was not automatically better: the original Random Forest achieved 62.13 km RMSE, while the optimised Random Forest produced 62.17 km. The original model was therefore retained.
6. **Freeze one common point estimator.** FNN achieved the lowest confirmatory 72-hour RMSE at 61.69 km and remained close to the best models on median error and ITE, so its predictions were carried unchanged into RQ2 and RQ3.

This sequence is important: it shows why the final model set contains both the original strong models and selected optimised variants rather than only the last version of every experiment.


## 2. Shared experimental boundary

- Each forecast case uses 24 hours of history to predict 72 hourly east–north displacements.
- Complete drifter IDs remain in one partition to prevent leakage between related windows.
- The frozen split contains 3,591 training, 769 calibration and 757 held-out test cases.
- Grouped cross-validation and model fitting use the training partition.
- Calibration is reserved for RQ3.
- RMSE is the primary ranking metric because large location misses matter operationally; median endpoint error and ITE are retained as complementary checks.


## 3. RQ1 Stage 1 — Two baselines and four initial learned models

The first round was intentionally broad rather than narrowly tuned. It asked which modelling assumptions were worth further effort.

| Initial model | Why it was tried | 72 h RMSE | What we learned |
|---|---|---:|---|
| Velocity persistence | Strong short-range physical reference using the last observed motion | 90.10 km | Recent motion alone was insufficient over 72 hours |
| Spatial mean flow | Location-dependent climatological reference | 80.98 km | Geographic flow structure helped, but individual recent history still mattered |
| Pooled VAR(1) | Test whether a shared linear velocity transition captured temporal dynamics | 77.89 km | Better than persistence, but too rigid for the full trajectory mapping |
| Ridge | Stable interpretable multi-output linear mapping | 65.91 km | A direct 144-output formulation produced a large gain over the dynamic baseline |
| Weighted KNN | Reuse future paths from similar historical cases | 65.21 km | Analogue matching was competitive and worth refining |
| FNN | Learn a nonlinear 98-to-144 dimensional trajectory mapping | 62.15 km | Best initial learned result and the clearest candidate for architecture and seed refinement |

The first screening changed the direction of the project. Rather than spending most of the remaining effort on increasingly complex AR models, we focused on flexible direct multi-output prediction.


## 4. RQ1 Stage 2 — Add tree ensembles

Tree models were added after the initial comparison because the first round suggested that nonlinear relationships were important.

| Added model | Motivation | Frozen result | Decision |
|---|---|---:|---|
| Extra Trees | Test highly randomised splits and robust averaging | 63.32 km RMSE | Retained as a strong nonlinear comparison |
| Original Random Forest | Test bootstrap aggregation with less randomised trees | **62.13 km RMSE** | Retained; immediately joined FNN in the leading performance tier |
| Physical-residual Extra Trees | Learn corrections to a physical forecast rather than the trajectory directly | Did not replace the leading direct models | Preserved as an informative hybrid-model trial |

Adding tree ensembles was useful even though a tree model was not selected as the final common estimator. Random Forest demonstrated that the FNN's advantage was small and that model choice depended on the loss function.


## 5. RQ1 Stage 3 — Targeted model strengthening

The strongest first-round ideas were not all optimised in the same way. Each received changes suited to its modelling assumption.

| Model path | What was strengthened | Before | After | Interpretation |
|---|---|---:|---:|---|
| Weighted KNN | Feature weighting, PCA variants and neighbour search | 65.21 km | 63.71 km | A meaningful improvement, but still above FNN and Random Forest |
| FNN | Hidden-layer architecture, regularisation, early stopping and seed ensembling | 62.15 km | **61.69 km** | The optimisation improved the primary metric and produced the lowest frozen RMSE |
| Random Forest | Wider grouped-CV search and larger ensembles | **62.13 km** | 62.17 km | Optimisation was slightly worse; the original Random Forest was kept |
| FNN seed audit | Three-seed versus four-seed ensemble | 61.69 km | 61.68 km | Essentially unchanged, supporting seed stability |

The Random Forest result is an important negative finding. More tuning did not guarantee better test performance. Reporting both values makes the selection process transparent and explains why the final comparison uses the original Random Forest rather than the nominally “optimised” version.


## 6. RQ1 Stage 4 — Trials that did not replace the finalists

| Trial | Question | Outcome and lesson |
|---|---|---|
| Higher-order VAR and ARIMA search | Would additional temporal lags improve pooled VAR(1)? | Added complexity did not justify replacing the simpler VAR(1) benchmark |
| Physical-residual Extra Trees | Would learning a correction around a physical model outperform direct trajectory prediction? | Competitive in places, but did not replace the leading direct models |
| Gradient-boosting residual correction | Could a second-stage residual model improve the selected forecast? | Reached 60.77 km after later held-out inspection; treated as exploratory and not used as confirmatory evidence |
| Alternative FNN seeds | Was the FNN result dependent on one random initialisation? | Three- and four-seed ensembles were virtually identical |
| Drogue diagnostic | Did attached versus lost drogue status explain the error difference? | Lost-drogue cases had raw errors about 10–11 km higher, but adjusted intervals crossed zero |

These trials are kept because they show what was tested, what failed to improve, and where the evidence was insufficient. They also prevent the final model choice from looking like a single successful run selected without alternatives.


## 7. Frozen RQ1 comparison and final choice

| Final retained model | 72 h RMSE | Role in the conclusion |
|---|---:|---|
| FNN | **61.69 km** | Lowest frozen RMSE; selected common point estimator |
| Original Random Forest | 62.13 km | Statistically competitive with FNN; original outperformed optimised RF |
| Extra Trees | 63.32 km | Strong tree-ensemble comparator |
| Optimised weighted KNN | 63.71 km | Strong analogue-based comparator |
| Ridge | 65.91 km | Best simple linear multi-output reference |
| Pooled VAR(1) | 77.89 km | Dynamic-linear reference |
| Spatial mean flow | 80.98 km | Physical climatology baseline |
| Velocity persistence | 90.10 km | Recent-motion physical baseline |

FNN was selected because it had the lowest numerical RMSE in the frozen comparison and remained close to the best model under median endpoint error and ITE. Random Forest was extremely close, and the paired interval crossed zero, so the report does not claim universal FNN superiority. The selection is a practical hand-off decision that gives RQ2 and RQ3 one consistent point forecast.


## 8. RQ2 — Regional predictability and spatial generalisation trials

| Development step | Purpose | Main lesson |
|---|---|---|
| Four-regime construction | Separate Shelf, Kuroshio Core, Transition and Other Offshore conditions | Revealed substantial geographic heterogeneity in forecast difficulty |
| Lead-time error curves by regime | Check when regional differences emerge | Kuroshio error grew fastest and remained the hardest regime |
| Regime-conditioned refitting | Test whether local models improved on the frozen global model | Local refitting did not provide one uniformly better strategy |
| Kuroshio-focused all-model extension | Check whether the hard regime changed model rankings | Reinforced that regional difficulty was not merely an FNN artefact |
| Absolute and normalised metrics | Separate kilometres of error from scale-relative difficulty | Both views were needed for a balanced interpretation |
| Zero-shot spatial transfer | Test models in geographically distinct external regions | Random Forest transferred most reliably; FNN and KNN degraded more strongly |

RQ2 therefore reused the frozen FNN for the primary regional question while keeping the spatial-transfer experiment as an external-validity analysis rather than silently changing the RQ1 estimator.


## 9. RQ3 — Uncertainty-calibration trials

| Development step | Purpose | Main result |
|---|---|---|
| Global lead-specific conformal calibration | Establish marginal prediction-region coverage | Global 90% coverage at 72 h was 91.02% with a 101.87 km radius |
| Mondrian regime calibration | Address conditional undercoverage | Kuroshio coverage improved from 80.58% to 89.93% |
| Multi-nominal evaluation | Check 50%, 80%, 90% and 95% targets | Showed how calibration quality and region size changed with the target level |
| Radius and area analysis | Quantify the operational cost of coverage | Better conditional coverage required larger Kuroshio regions |
| Paired bootstrap | Test the stability of Global–Mondrian differences | Kuroshio improvement was +9.35 percentage points, with a positive paired interval |
| Validation audit | Check package completeness and reproducibility | 19 PASS, 2 NOT_VERIFIED and 0 FAIL |

RQ3 did not retrain the point model. It calibrated uncertainty around the same frozen FNN forecasts, preserving a clear separation between point accuracy and uncertainty quality.


## 10. Notebook provenance index

The links below connect each modelling step to its organised English-facing notebook.


### RQ1

- [08_model_error_and_baseline_comparison](../RQ1/notebooks/analysis/08_model_error_and_baseline_comparison.ipynb) — analysis
- [09_each_model_vs_two_baselines](../RQ1/notebooks/analysis/09_each_model_vs_two_baselines.ipynb) — analysis
- [14_fnn_seed_ensemble_audit](../RQ1/notebooks/analysis/14_fnn_seed_ensemble_audit.ipynb) — analysis
- [15_champion_models_drogue_effect](../RQ1/notebooks/analysis/15_champion_models_drogue_effect.ipynb) — analysis
- [RQ1_training_and_test_audit](../RQ1/notebooks/analysis/RQ1_training_and_test_audit.ipynb) — analysis
- [01_velocity_persistence_baseline](../RQ1/notebooks/core/01_velocity_persistence_baseline.ipynb) — core
- [02_spatial_mean_flow_baseline](../RQ1/notebooks/core/02_spatial_mean_flow_baseline.ipynb) — core
- [03_multioutput_ridge_trajectory_model](../RQ1/notebooks/core/03_multioutput_ridge_trajectory_model.ipynb) — core
- [04_fnn_trajectory_model](../RQ1/notebooks/core/04_fnn_trajectory_model.ipynb) — core
- [05_pooled_var1_velocity_model](../RQ1/notebooks/core/05_pooled_var1_velocity_model.ipynb) — core
- [06_weighted_knn_trajectory_regression](../RQ1/notebooks/core/06_weighted_knn_trajectory_regression.ipynb) — core
- [07_ARIMA_VAR_model_search](../RQ1/notebooks/extensions/07_ARIMA_VAR_model_search.ipynb) — extensions
- [07_extra_trees_trajectory_model](../RQ1/notebooks/extensions/07_extra_trees_trajectory_model.ipynb) — extensions
- [08_random_forest_trajectory_model](../RQ1/notebooks/extensions/08_random_forest_trajectory_model.ipynb) — extensions
- [09_physical_residual_extra_trees](../RQ1/notebooks/extensions/09_physical_residual_extra_trees.ipynb) — extensions
- [10_optimised_random_forest](../RQ1/notebooks/extensions/10_optimised_random_forest.ipynb) — extensions
- [11_optimised_fnn](../RQ1/notebooks/extensions/11_optimised_fnn.ipynb) — extensions
- [12_optimised_weighted_knn](../RQ1/notebooks/extensions/12_optimised_weighted_knn.ipynb) — extensions
- [13_gradient_boosting_residual](../RQ1/notebooks/extensions/13_gradient_boosting_residual.ipynb) — extensions
- [14_champion_models_comparison](../RQ1/notebooks/extensions/14_champion_models_comparison.ipynb) — extensions


### RQ2

- [02_champion_models_spatial_generalization](../RQ2/notebooks/champion_models/02_champion_models_spatial_generalization.ipynb) — champion_models
- [RQ2_champion_models](../RQ2/notebooks/champion_models/RQ2_champion_models.ipynb) — champion_models
- [20_regional_lead_time_analysis](../RQ2/notebooks/core/20_regional_lead_time_analysis.ipynb) — core
- [RQ2_complete_analysis](../RQ2/notebooks/core/RQ2_complete_analysis.ipynb) — core
- [RQ2_step_by_step](../RQ2/notebooks/core/RQ2_step_by_step.ipynb) — core
- [01_region_selection_and_cache](../RQ2/notebooks/extensions/01_region_selection_and_cache.ipynb) — extensions
- [18_regime_conditioned_forecasting](../RQ2/notebooks/extensions/18_regime_conditioned_forecasting.ipynb) — extensions
- [19_Kuroshio](../RQ2/notebooks/extensions/19_Kuroshio.ipynb) — extensions
- [21_teacher_feedback_normalised_evaluation](../RQ2/notebooks/extensions/21_teacher_feedback_normalised_evaluation.ipynb) — extensions


### RQ3

- [RQ3_all_report_figures_source](../RQ3/notebooks/analysis/RQ3_all_report_figures_source.ipynb) — analysis
- [RQ3_multi_nominal_calibration](../RQ3/notebooks/analysis/RQ3_multi_nominal_calibration.ipynb) — analysis
- [RQ3_champion_conformal](../RQ3/notebooks/champion_models/RQ3_champion_conformal.ipynb) — champion_models
- [RQ3_complete_analysis](../RQ3/notebooks/core/RQ3_complete_analysis.ipynb) — core
- [RQ3_conformal_prediction](../RQ3/notebooks/core/RQ3_conformal_prediction.ipynb) — core
- [RQ3_optimised_fnn_final](../RQ3/notebooks/core/RQ3_optimised_fnn_final.ipynb) — core


### 00_shared

- [01_data_check_clean](notebooks/preliminary/01_data_check_clean.ipynb) — preliminary
- [explore](notebooks/preliminary/explore.ipynb) — preliminary
- [project_progress_overview](notebooks/preliminary/project_progress_overview.ipynb) — preliminary
- [proposal_method_and_initial_analysis](notebooks/preliminary/proposal_method_and_initial_analysis.ipynb) — preliminary
- [regionA_final](notebooks/preliminary/regionA_final.ipynb) — preliminary
- [01_sections_3_4_three_minute_script](notebooks/presentation/01_sections_3_4_three_minute_script.ipynb) — presentation
- [Random_Forest_RMSE_Optimisation_Audit](notebooks/presentation/Random_Forest_RMSE_Optimisation_Audit.ipynb) — presentation


## 11. Interpretation and reproducibility status

The organised repository preserves both successful and unsuccessful model-development paths. Numerical claims in the final report come from the frozen comparison unless explicitly labelled exploratory. Later experiments remain visible because they document effort and inform future work, but they do not overwrite the confirmatory result after the held-out test set had been inspected.
